## Business Problem Statement

Customers are not currently segmented by value or retention risk, so marketing and retention efforts are applied uniformly rather than targeted. This creates two costs: wasted spend retaining low-risk customers, and undetected disengagement among high-value customers until revenue impact is already visible.

Objective: Segment customers by value and behavior, identify those at risk of churn, and quantify the revenue tied to that risk — enabling retention efforts to be prioritized where they matter most.

## 1. Data Loading

Loading both sheets from the Online Retail II dataset (Year 2009-2010 and 
Year 2010-2011) and combining them into a single dataframe.

In [1]:
import pandas as pd

df = pd.read_csv('RFM Data/Raw dataset/rfm_combined.csv')

print(df.shape)

(1067371, 8)


In [2]:
df.dtypes

Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object

In [3]:
df.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

## 2. Handling Nulls
#### Dropping nulls in customer_id since RFM analysis needs customer_id 
#### Renaming nulls in description 

In [5]:
df_rfm=df.dropna(subset=['Customer ID'])
print(df_rfm.shape)

(824364, 8)


In [6]:
df['Description']=df['Description'].fillna('Unknown')

In [7]:
df_rfm.isnull().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

In [8]:
df_rfm[df_rfm['Quantity']<0].shape

(18744, 8)

In [9]:
df_rfm[df_rfm['Quantity']<0]['Invoice'].astype(str).str.startswith('C').value_counts()

Invoice
True    18744
Name: count, dtype: int64

In [10]:
print(df_rfm[df_rfm['Price']<=0].shape)
print(df_rfm.duplicated().sum())


(71, 8)
26479


#### Dropping Duplicates

In [11]:
df_rfm=df_rfm.drop_duplicates()
print(df_rfm.shape)

(797885, 8)


#### Dropping Negative Prices

In [12]:
df_rfm=df_rfm[df_rfm['Price']>0]
print(df_rfm.shape)

(797815, 8)


In [13]:
print(df_rfm['InvoiceDate'].min())
print(df_rfm['InvoiceDate'].max())

2009-12-01 07:45:00
2011-12-09 12:50:00


In [14]:
df_rfm['InvoiceDate']=pd.to_datetime(df_rfm['InvoiceDate'])

In [15]:
reference_date=df_rfm['InvoiceDate'].max()+pd.Timedelta(days=1)
print(reference_date)

2011-12-10 12:50:00


In [16]:
df_rfm['Total Amount']=df_rfm['Quantity']*df_rfm['Price']

In [17]:
df_rfm[['Quantity','Price','Total Amount']].head()

,Quantity,Price,Total Amount
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


## RFM

In [18]:
rfm=df_rfm.groupby('Customer ID').agg(
    Recency=('InvoiceDate',lambda x:(reference_date-x.max()).days),
    Frequency=('Invoice','nunique'),
    Monetary=('Total Amount','sum')
).reset_index()

rfm.head()
    

,Customer ID,Recency,Frequency,Monetary
0,12346.0,326,17,-51.74
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,19,5,4404.54
4,12350.0,310,1,334.40


#### Recency

In [19]:
rfm['R_score']=pd.qcut(rfm['Recency'],5,labels =[5,4,3,2,1])

#### Frequency

In [20]:
rfm['F_score']=pd.qcut(rfm['Frequency'].rank(method='first'),5,labels=[1,2,3,4,5])

#### Monetary

In [21]:
rfm['M_score']=pd.qcut(rfm['Monetary'],5,labels=[1,2,3,4,5])

In [22]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_score,F_score,M_score
0,12346.0,326,17,-51.74,2,5,1
1,12347.0,2,8,4921.53,5,4,5
2,12348.0,75,5,2019.40,3,3,4
3,12349.0,19,5,4404.54,4,3,5
4,12350.0,310,1,334.40,2,1,2


In [23]:
rfm['RFM_Score']=rfm['R_score'].astype(str)+rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

In [24]:
def rfm_segment (row):
    r,f,m =row['R_score'],row['F_score'],row['M_score']
    r,f,m=int(r),int(f),int(m)

    if r>=4 and f>=4 and m>=4:
        return 'High Value'
    elif r>=3 and f>=3 and m>=3:
        return 'Loyal'
    elif r>=4 and f<=2 :
        return 'New'
    elif r>=3 and f<=2 and m<=2:
        return 'Growing'
    elif r<=2 and f>=4 and m>=4:
        return 'At Risk'
    elif r<=2 and f>=3 and m<=2:
        return 'Needs Win-Back'
    elif r<=2 and f<=2 and m<=2:
        return 'Low Value'  
    elif r>=2 and f>=2 and m>=2:
        return 'Steady'
    elif r <= 2 and f >= 2 and m >= 2:
        return 'Fading'    
    else:
        return 'Uncategorized'

rfm['Segment']=rfm.apply(rfm_segment,axis=1)


In [25]:
rfm['Segment'].value_counts()

Segment
High Value        1303
Low Value         1301
Loyal             1187
Steady             600
New                459
Growing            288
At Risk            226
Needs Win-Back     218
Fading             205
Uncategorized      152
Name: count, dtype: int64

In [26]:
rfm[rfm['Segment'] == 'Uncategorized'][['R_score','F_score','M_score']].value_counts().head(10)

R_score  F_score  M_score
1        1        3          34
2        1        3          26
3        3        1          20
         1        3          19
5        3        1          11
4        3        1          11
2        1        4           7
3        1        4           6
1        1        4           5
4        4        1           4
Name: count, dtype: int64

In [27]:
rfm.to_csv('RFM Data/Raw dataset/rfm_scored.csv',index=False)

In [28]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score,Segment
0,12346.0,326,17,-51.74,2,5,1,251,Needs Win-Back
1,12347.0,2,8,4921.53,5,4,5,545,High Value
2,12348.0,75,5,2019.40,3,3,4,334,Loyal
3,12349.0,19,5,4404.54,4,3,5,435,Loyal
4,12350.0,310,1,334.40,2,1,2,212,Low Value


In [29]:
import os; print(os.getcwd())

C:\Users\heraa\RFM  Project


In [31]:
C:\Users\heraa\RFM  Project\RFM Data\Raw dataset\rfm_scored.csv



SyntaxError: unexpected character after line continuation character (2633237514.py, line 1)

In [32]:
df_rfm.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country', 'Total Amount'],
      dtype='object')

In [34]:
df_rfm.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total Amount
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [35]:
df_rfm['InvoiceDate'].dtype

dtype('<M8[ns]')

In [36]:
len(df_rfm)


797815

In [37]:
df_rfm.to_csv('RFM Data/Raw dataset/df_rfm.csv', index=False)